# Pertemuan 1: Regresi linear dengan least squares

[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dafx7/COMPUTING-AI-ML-OPTIMIZATION/blob/main/week1/notebooks/01_least_squares.ipynb)

Hari ini kita menulis rumus least squares dalam **Python murni**, tanpa NumPy dan tanpa pustaka ML.

**Cara kerja**
1. Tekan `Shift + Enter` untuk menjalankan sel kode, dari atas ke bawah.
2. Ganti setiap `...` dengan kodemu, lalu jalankan sel pemeriksa di bawahnya. Pemeriksa mencetak `Benar.` atau petunjuk.
3. Kamu boleh memakai asisten AI. Pastikan kamu bisa menjelaskan setiap baris yang kamu kumpulkan: kode ini akan ditanyakan di cek lisan Pekan 8.

**Isi**
- Bagian A: Python secukupnya untuk programmer Go (sekitar 15 menit)
- Latihan 4: rumus least squares, diuji pada data Go-car, lalu dipakai pada data baterai
- Latihan 5: merusak rumusnya
- Dua pertanyaan pemahaman (dikumpulkan)

In [ ]:
def cek(syarat, pesan="Belum tepat."):
    """Memeriksa jawabanmu. `syarat` adalah fungsi tanpa argumen yang mengembalikan True atau False."""
    try:
        benar = bool(syarat())
    except Exception as e:
        benar = False
        pesan = f"{pesan} (Kode belum lengkap atau menimbulkan error: {type(e).__name__}: {e})"
    print("Benar." if benar else "Belum tepat. " + pesan)


def dekat(a, b, toleransi=1e-6):
    """True jika a dan b hampir sama (selisih relatif kecil)."""
    return abs(a - b) <= toleransi * max(1.0, abs(b))

## Bagian A: Python secukupnya

### A1. Variabel dan pembagian

**Go**
```go
total := 97 + 89 + 80
rata := float64(total) / 3
```
**Python** tidak memakai deklarasi tipe, dan `/` selalu menghasilkan bilangan pecahan (float).
Hitung rata-rata dari tiga catatan baterai: 97, 89, dan 80.

In [ ]:
total = 97 + 89 + 80
rata = total / 3
print(rata)

In [ ]:
cek(lambda: total == 266 and dekat(rata, 266 / 3), 'total seharusnya 266 dan rata seharusnya 266 / 3.')

### A2. List: padanan slice di Go

**Go**
```go
baterai := []int{97, 89, 80, 74}
baterai = append(baterai, 66)
n := len(baterai)
pertama := baterai[0]
```
**Python** memakai list dengan kurung siku. Indeks mulai dari 0, dan `daftar[-1]` adalah elemen terakhir.
Tambahkan 66 ke list, lalu isi `n`, `pertama`, dan `terakhir`.

In [ ]:
baterai = [97, 89, 80, 74]
baterai.append(66)
n = len(baterai)
pertama = baterai[0]
terakhir = baterai[-1]

In [ ]:
cek(lambda: baterai == [97, 89, 80, 74, 66] and n == 5 and pertama == 97 and terakhir == 66, 'Setelah append, list berisi 5 angka, pertama 97, terakhir 66.')

### A3. Perulangan for

**Go**
```go
jumlah := 0
for _, v := range baterai {
    jumlah += v
}
```
**Python**
```python
for v in baterai:
    ...
```
Blok ditandai dengan **indentasi** (4 spasi), tanpa kurung kurawal. Hitung jumlah semua nilai di `baterai` dengan perulangan.

In [ ]:
jumlah = 0
for v in baterai:
    jumlah += v
print(jumlah)

In [ ]:
cek(lambda: jumlah == 406, 'Jumlah 97 + 89 + 80 + 74 + 66 adalah 406.')

### A4. sum() dan list comprehension

Python punya `sum(daftar)` bawaan. Untuk membuat list baru dari list lama, ada **list comprehension**:
```python
kuadrat = [v * v for v in daftar]
```
Padanan Go-nya adalah perulangan yang mengisi slice baru. Hitung `jumlah2` dengan `sum`, dan buat `selisih` berisi setiap nilai dikurangi 80.

In [ ]:
jumlah2 = sum(baterai)
selisih = [v - 80 for v in baterai]
print(jumlah2, selisih)

In [ ]:
cek(lambda: jumlah2 == 406 and selisih == [17, 9, 0, -6, -14], 'selisih seharusnya [17, 9, 0, -6, -14].')

### A5. zip: berjalan di dua list sekaligus

**Go**
```go
for i := range x {
    total += x[i] * y[i]
}
```
**Python** bisa memasangkan dua list dengan `zip`:
```python
for xi, yi in zip(x, y):
    ...
```
Hitung jumlah dari `xi * yi`.

In [ ]:
x = [0, 10, 20, 30, 40]
y = [97, 89, 80, 74, 66]
total_xy = 0
for xi, yi in zip(x, y):
    total_xy += xi * yi
print(total_xy)

In [ ]:
cek(lambda: total_xy == 7350, '0*97 + 10*89 + 20*80 + 30*74 + 40*66 = 7350.')

### A6. Fungsi

**Go**
```go
func rataRata(d []float64) float64 {
    ...
}
```
**Python**
```python
def rata_rata(daftar):
    return ...
```
Tidak ada tipe untuk parameter maupun nilai kembali. Tulis `rata_rata` yang mengembalikan rata-rata sebuah list. Fungsi ini akan kamu pakai lagi di Latihan 4.

In [ ]:
def rata_rata(daftar):
    return sum(daftar) / len(daftar)

In [ ]:
cek(lambda: dekat(rata_rata([97, 89, 80, 74, 66]), 81.2) and dekat(rata_rata([5, 5, 6]), 16 / 3), 'rata_rata([97, 89, 80, 74, 66]) seharusnya 81.2.')

## Latihan 4: rumus least squares

Kita uji dulu pada data Go-car, karena jawabannya sudah kita lihat di slide: ŵ ≈ 5737,7 dan â ≈ 3148,8. Jika kodemu memberi angka yang sama, kodemu benar.

Notasi: model `f(x) = w·x + a`, dengan `w` kemiringan dan `a` titik potong. RSS (residual sum of squares) di materi kuliah disebut juga SSE. Di Python kita menulis `x_bar` untuk x̄ (rata-rata x).

In [ ]:
# Data Go-car dari slide (15 perjalanan). Contoh: Dr. Didit Adytia, S.Si., M.Si.
km    = [5, 5, 6, 6, 8, 8, 9, 9, 10, 11, 11, 12, 12, 12, 13]
tarif = [28600, 40100, 32600, 38600, 49700, 46100, 55300, 49800, 58700, 67700, 72600, 79800, 72400, 70600, 70700]
print(len(km), "perjalanan")

### 4a. Kemiringan ŵ, persamaan (7)

$$\hat{w} = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Tulis `hitung_w(x, y)`. Pakai `rata_rata` dari A6. Petunjuk: hitung pembilang dan penyebut dengan perulangan `zip` atau dengan `sum` dan list comprehension.

In [ ]:
def hitung_w(x, y):
    x_bar = rata_rata(x)
    y_bar = rata_rata(y)
    pembilang = sum((xi - x_bar) * (yi - y_bar) for xi, yi in zip(x, y))
    penyebut = sum((xi - x_bar) ** 2 for xi in x)
    return pembilang / penyebut

In [ ]:
cek(lambda: dekat(hitung_w(km, tarif), 5737.724935732648), 'Untuk data Go-car, ŵ seharusnya sekitar 5737.72 (lihat slide).')

### 4b. Titik potong â, persamaan (4)

$$\hat{a} = \bar{y} - w\,\bar{x}$$

In [ ]:
def hitung_a(x, y, w):
    return rata_rata(y) - w * rata_rata(x)

In [ ]:
cek(lambda: dekat(hitung_a(km, tarif, hitung_w(km, tarif)), 3148.778920308483), 'Untuk data Go-car, â seharusnya sekitar 3148.78.')

### 4c. Prediksi dan RSS

$$\hat{y} = w\,x + a \qquad \mathrm{RSS}(w,a) = \sum_{i=1}^{n} \left(y_i - w\,x_i - a\right)^2$$

In [ ]:
def prediksi(x_baru, w, a):
    return w * x_baru + a

def rss(x, y, w, a):
    return sum((yi - prediksi(xi, w, a)) ** 2 for xi, yi in zip(x, y))

In [ ]:
cek(lambda: dekat(rss(km, tarif, 5040, 8800), 355166000) and dekat(prediksi(20, 2, 1), 41), 'RSS garis B, y = 8800 + 5040x, pada data Go-car seharusnya 355166000.')

### 4d. Data baterai

Sekarang pakai fungsimu pada 19 catatan baterai. Hubungannya negatif, jadi ŵ harus negatif.

Lalu cocokkan dengan pustaka standar Python: `statistics.linear_regression` (tersedia sejak Python 3.10) menghitung garis least squares yang sama. Angkanya harus sama dengan milikmu.

In [ ]:
# Data latihan: baterai HP (%) selama bermain game, dicatat setiap 5 menit.
# Ini data contoh untuk latihan, bukan hasil pengukuran.
menit   = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90]
baterai = [100, 93, 90, 84, 80, 76, 73, 68, 65, 60, 56, 53, 49, 43, 40, 35, 33, 27, 23]
print(len(menit), "catatan")

In [ ]:
w_bat = hitung_w(menit, baterai)
a_bat = hitung_a(menit, baterai, w_bat)
print(f"w = {w_bat:.4f}   a = {a_bat:.4f}   RSS = {rss(menit, baterai, w_bat, a_bat):.2f}")

import statistics
hasil = statistics.linear_regression(menit, baterai)
print(f"statistics: slope = {hasil.slope:.4f}   intercept = {hasil.intercept:.4f}")

In [ ]:
cek(lambda: dekat(w_bat, hasil.slope) and dekat(a_bat, hasil.intercept), 'Hasilmu harus sama dengan statistics.linear_regression.')

### 4e. Kapan baterai habis?

Baterai habis saat prediksi bernilai 0: `w·x + a = 0`. Selesaikan untuk `x`, lalu hitung juga prediksi untuk 180 menit.

In [ ]:
menit_habis = -a_bat / w_bat
prediksi_180 = prediksi(180, w_bat, a_bat)
print(f"Habis setelah sekitar {menit_habis:.0f} menit. Prediksi 180 menit: {prediksi_180:.1f}%")

In [ ]:
cek(lambda: dekat(menit_habis, -a_bat / w_bat) and dekat(prediksi_180, w_bat * 180 + a_bat), 'menit_habis adalah -a / w.')

## Latihan 5: rusak rumusnya

### 5a. Semua catatan di menit yang sama
Lima teman mencatat baterai HP masing-masing, semuanya tepat di menit 30. Jalankan sel di bawah. Error apa yang muncul, dan **baris mana di rumus (7)** yang menyebabkannya?

In [ ]:
x_sama = [30, 30, 30, 30, 30]
y_sama = [74, 70, 77, 72, 75]
try:
    print(hitung_w(x_sama, y_sama))
except Exception as e:
    print("Error:", type(e).__name__, e)

*Kunci:* `ZeroDivisionError`. Semua `xᵢ` sama dengan x̄, jadi penyebut Σ(xᵢ − x̄)² bernilai 0. Tanpa variasi di x, kemiringan tidak terdefinisi: data tidak memberi tahu apa pun tentang pengaruh menit terhadap baterai.

### 5b. Satu salah ketik
Catatan di menit 40 seharusnya 65, tetapi terketik 6. Buat salinan list, ubah satu nilai itu, lalu hitung ulang ŵ dan â. Bandingkan dengan hasil 4d.

In [ ]:
baterai_salah = list(baterai)        # salinan, supaya data asli tidak berubah
i = menit.index(40)
baterai_salah[i] = 6
w_salah = hitung_w(menit, baterai_salah)
a_salah = hitung_a(menit, baterai_salah, w_salah)
print(f"asli : w = {w_bat:.3f}   a = {a_bat:.2f}")
print(f"salah: w = {w_salah:.3f}   a = {a_salah:.2f}")

In [ ]:
cek(lambda: baterai_salah[i] == 6 and baterai[i] == 65 and dekat(w_salah, hitung_w(menit, baterai_salah)) and dekat(a_salah, hitung_a(menit, baterai_salah, w_salah)), 'Ubah baterai_salah, jangan baterai. Lalu hitung w_salah dan a_salah dari baterai_salah.')

*Kunci:* ŵ hampir tidak berubah (−0,827 menjadi −0,806), tetapi â turun dari sekitar 97,6 menjadi 93,6. Error dikuadratkan, jadi satu titik yang meleset 59 poin menyumbang sangat besar ke RSS, dan garis bergeser untuk mengurangi error titik itu. Least squares peka terhadap pencilan (outlier).

## Pertanyaan pemahaman: contoh jawaban

1. RSS berbentuk mangkuk terhadap w dan a. Di titik terendah, kemiringan (turunan parsial) terhadap w dan terhadap a sama dengan nol. Rumus ŵ dan â adalah hasil menyelesaikan dua persamaan "turunan = 0" itu, sehingga tidak ada pasangan w dan a lain dengan RSS lebih kecil.
2. ŵ negatif berarti setiap tambahan satu menit bermain mengurangi baterai sekitar 0,83 persen. Prediksi 180 menit memberi angka negatif karena garis diperpanjang jauh di luar data (0 sampai 90 menit); baterai tidak bisa di bawah 0 persen, dan pengurasan baterai sungguhan juga tidak persis lurus.